# 01 – Data
Goal: load the perpetual we will actually trade, validate it, cache it. Checks: gaps, duplicates, closed candles only, UTC. Add funding rates.

**Why this notebook comes first:** every later result (baselines, models, validation) is only as trustworthy as this series. Research must use the instrument that will be traded (README §1).

**Data terms (read before running):** Deribit's Terms of Service allow market data for *personal use only*; no aggregating, reselling, publishing or forwarding (evidence E34, 2021 version of the terms: re-read the current one on deribit.com). The cache in `data/` is git-ignored and must stay private. Do not commit or share it.

**Needs network access to www.deribit.com** (public endpoints, no API key). Run locally, not in the sandbox.

Status: in progress

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from mlt.data.cache import FrameCache
from mlt.data.providers.deribit import DeribitProvider
from mlt.data.schema import find_gaps, validate_candles

INSTRUMENT = "BTC_USDC-PERPETUAL"   # the product shown in the Coinbase account (ADR 0003, to verify below)
INTERVAL = "1h"
END = pd.Timestamp.now(tz="UTC").floor("1h")
START = END - pd.Timedelta(days=3 * 365)

provider = DeribitProvider()
cache = FrameCache("../data")       # git-ignored, private

## 1. Contract specification
ADR 0003 hypothesis: `BTC_USDC-PERPETUAL` on Coinbase Global Derivatives is the Deribit contract of the same name, so Deribit public candles are the traded series. Check the specification and compare it **by hand** with the product page in Coinbase Advanced (contract size, tick size, funding interval, fees, max leverage).

In [ ]:
spec = provider.get_instrument(INSTRUMENT)
keys = ["instrument_name", "kind", "instrument_type", "settlement_period", "quote_currency",
        "base_currency", "contract_size", "tick_size", "min_trade_amount",
        "maker_commission", "taker_commission", "max_leverage", "creation_timestamp"]
pd.Series({k: spec.get(k) for k in keys})

**Manual comparison (fill in after looking at Coinbase):**

| Field | Deribit (above) | Coinbase product page | Match? |
|---|---|---|---|
| contract size | | | |
| tick size | | | |
| settlement / funding | | | |
| taker / maker fee | | | |

If anything differs materially, the series is a proxy and ADR 0003 must say how large the difference is before notebooks 02–04 are trusted.

## 2. Candles (cached, closed candles only)

In [ ]:
candles = cache.get_candles(provider, INSTRUMENT, INTERVAL, START, END)
print(candles.shape, candles.index[0], "->", candles.index[-1])
candles.tail()

## 3. Quality checks

In [ ]:
validate_candles(candles)                       # UTC, sorted, unique, positive prices, high >= low
gaps = find_gaps(candles, INTERVAL)             # reported, never filled
print("gaps:", len(gaps), "missing candles:", int(gaps["n_missing"].sum()) if len(gaps) else 0)
print("zero-volume candles:", int((candles["volume"] == 0).sum()))
print("duplicate timestamps:", int(candles.index.duplicated().sum()))
bad = candles[(candles["high"] < candles[["open", "close"]].max(axis=1))
              | (candles["low"] > candles[["open", "close"]].min(axis=1))]
print("candles where open/close lie outside high-low:", len(bad))
gaps.head(20)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
candles["close"].plot(ax=ax[0], title=f"{INSTRUMENT} close")
candles["volume"].plot(ax=ax[1], title="volume", color="gray")
plt.tight_layout()

Decide here how to treat any gaps found (drop the affected windows, or shorten the period). **Never fill them silently.** Record the decision under *Findings*.

## 4. Funding

In [ ]:
funding = cache.get_funding(provider, INSTRUMENT, START, END)
print(funding.shape, funding.index[0], "->", funding.index[-1])
print("mean interest_1h:", funding["interest_1h"].mean(), " std:", funding["interest_1h"].std())
print("approx. annualised (mean interest_1h * 24 * 365):", funding["interest_1h"].mean() * 24 * 365)
funding.tail()

In [ ]:
# Funding must line up with the candle grid before it enters a backtest.
aligned = funding["interest_1h"].reindex(candles.index)
print("candles without a funding observation:", int(aligned.isna().sum()), "of", len(candles))

## Findings (fill in after running; only numbers actually printed above)
- Period and number of candles:
- Gaps / zero-volume candles and how they are handled:
- Specification match with the Coinbase product (yes / no, details):
- Funding: coverage, alignment with candles, unusual values:
- Conclusion for ADR 0003 (proxy or the traded series?):

When this notebook is complete: set `Status: done`, add an `R-001` entry to `docs/RESEARCH_LOG.md` with the numbers, and update ADR 0003 with `Evidence:` pointing here.